In [14]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier

df = pd.read_csv('../data/diabetes.csv')
print("โหลดข้อมูล Diabetes เรียบร้อย!")
df.head()

โหลดข้อมูล Diabetes เรียบร้อย!


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [15]:
# ค่า 0 ในคอลัมน์เหล่านี้คือข้อมูลที่ขาดหาย (Missing Values)
cols_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_fix:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].mean())

In [16]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

scaler_dia = StandardScaler()
X_scaled = scaler_dia.fit_transform(X)

m1 = RandomForestClassifier(n_estimators=100, random_state=42)
m2 = AdaBoostClassifier(n_estimators=100, random_state=42)
m3 = LogisticRegression()

ensemble = VotingClassifier(estimators=[('rf', m1), ('ada', m2), ('lr', m3)], voting='soft')
ensemble.fit(X_scaled, y)

pickle.dump(ensemble, open('../models/diabetes_model.pkl', 'wb'))

pickle.dump(scaler_dia, open('../models/scaler_diabetes.pkl', 'wb'))

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# แบ่งข้อมูลก่อน scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# scale ใหม่ (ไม่กระทบของเดิม)
scaler_temp = StandardScaler()
X_train_scaled = scaler_temp.fit_transform(X_train)
X_test_scaled = scaler_temp.transform(X_test)

# ใช้โมเดลใหม่สำหรับ test เท่านั้น
ensemble_temp = VotingClassifier(
    estimators=[('rf', m1), ('ada', m2), ('lr', m3)],
    voting='soft'
)

ensemble_temp.fit(X_train_scaled, y_train)
y_pred = ensemble_temp.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy (test set): {acc*100:.2f}%")

Accuracy (test set): 75.97%
